<a href="https://colab.research.google.com/github/FattahulAlim/Machine_Learning_Sem5/blob/main/JS06/Praktikum6_244107020018_MFattahulAlim.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install annoy
!pip install faiss-cpu
!pip install hnswlib

In [ ]:
import pandas as pd
import numpy as np
import time
import faiss
from annoy import AnnoyIndex
import hnswlib
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler

from google.colab import drive
drive.mount('/content/drive')

# -------------------------------
# Load dataset
# -------------------------------
df = pd.read_csv('/content/drive/MyDrive/ml-sem5/JS06/dataset/songs_with_attributes_and_lyrics.csv')  # ganti path sesuai lokasi file
features = ['danceability', 'energy', 'loudness', 'speechiness',
            'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo']
X = df[features].values

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# Standarisasi fitur
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

k = 10  # jumlah nearest neighbors

In [ ]:
# -------------------------------
# Exact Nearest Neighbor (brute-force)
# -------------------------------
start = time.time()
nn = NearestNeighbors(n_neighbors=k, algorithm='brute', metric='euclidean')
nn.fit(X_scaled)
dist_exact, idx_exact = nn.kneighbors(X_scaled)
time_exact = time.time() - start
print(f"Exact NN done in {time_exact:.3f} s")

Exact NN done in 5183.751 s


In [ ]:
# -------------------------------
# Annoy
# -------------------------------
start = time.time()
f = X_scaled.shape[1]
index_annoy = AnnoyIndex(f, 'euclidean')
for i, v in enumerate(X_scaled):
    index_annoy.add_item(i, v)
index_annoy.build(10)
idx_annoy = [index_annoy.get_nns_by_vector(v, k) for v in X_scaled]
time_annoy = time.time() - start
print(f"Annoy done in {time_annoy:.3f} s")


Annoy done in 79.005 s


In [ ]:
# -------------------------------
# HNSW
# -------------------------------
start = time.time()
p_hnsw = hnswlib.Index(space='l2', dim=X_scaled.shape[1])
p_hnsw.init_index(max_elements=X_scaled.shape[0], ef_construction=200, M=16)
p_hnsw.add_items(X_scaled)
p_hnsw.set_ef(200)
idx_hnsw, dist_hnsw = p_hnsw.knn_query(X_scaled, k=k)
time_hnsw = time.time() - start
print(f"HNSW done in {time_hnsw:.3f} s")

HNSW done in 384.665 s


In [ ]:
# -------------------------------
# FAISS IVF
# -------------------------------
start = time.time()

X_faiss = np.ascontiguousarray(
    X_scaled, dtype=np.float32
)

d = X_faiss.shape[1]
nlist = 100

quantizer = faiss.IndexFlatL2(d)

index_faiss = faiss.IndexIVFFlat(
    quantizer, d, nlist, faiss.METRIC_L2
)

index_faiss.train(X_faiss)
index_faiss.add(X_faiss)
index_faiss.nprobe = 10

dist_faiss, idx_faiss = index_faiss.search(
    X_faiss, k
)

time_faiss = time.time() - start
print(f"FAISS IVF done in {time_faiss:.3f} s")

FAISS IVF done in 1555.799 s


In [ ]:
# -------------------------------
# Contoh tampilkan top-5 neighbors dari item pertama
# -------------------------------
print("\nTop-5 neighbors for first song:")
print(f"Exact NN: {idx_exact[0][:5]}")
print(f"Annoy:    {idx_annoy[0][:5]}")
print(f"HNSW:     {idx_hnsw[0][:5]}")
print(f"FAISS:    {idx_faiss[0][:5]}")


Top-5 neighbors for first song:
Exact NN: [     0 394553 764272 837727 749223]
Annoy:    [0, 764272, 749223, 625227, 833164]
HNSW:     [     0 394553 764272 837727 749223]
FAISS:    [     0 394553 764272 837727 749223]


# Analisis

Berdasarkan pengujian Top-5 tetangga terdekat pada lagu pertama, HNSW dan FAISS IVF memberikan hasil yang identik dengan Exact NN, sedangkan Annoy memiliki tiga indeks tetangga yang sama dari lima indeks yang dibandingkan. Dengan demikian, HNSW dan FAISS IVF menunjukkan kesesuaian hasil terbaik pada pengujian ini.

Namun, hasil tersebut hanya menggambarkan satu lagu sebagai contoh, sehingga belum cukup untuk menentukan algoritma terbaik secara keseluruhan. Evaluasi yang lebih menyeluruh perlu mempertimbangkan kesesuaian tetangga pada seluruh dataset, waktu pencarian, dan penggunaan memori.

Selain itu, karena data menggunakan sembilan atribut audio yang telah distandardisasi, tetangga terdekat ditentukan berdasarkan kemiripan jarak Euclidean dalam ruang fitur tersebut, bukan berdasarkan kemiripan lirik atau preferensi musik pendengar.